<a href="https://colab.research.google.com/github/EMatCM/proyectoAprendizaje/blob/main/notebooks/baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**¡¡REVISAR LOS PARAMETROS DE LOS INTERRUPTORES PARA EVITAR DESCARGAR EL DATASET O HACERLO DE SER NECESARIO!!**

In [ ]:
#clonar datos del repositorio
import os
REPO = "/content/proyectoAprendizaje"
if not os.path.exists(REPO):
    !git clone https://github.com/EMatCM/proyectoAprendizaje.git {REPO}
%cd {REPO}
!git pull
!ls modelos resultados

/content/proyectoAprendizaje
Already up to date.
modelos:
baseline_v1.pt

resultados:
baseline_test.json


In [ ]:
!pip install -q torch torchvision pillow

import json, random, time
from pathlib import Path

import torch
import torch.nn as nn
from PIL import Image
from torchvision import datasets, models, transforms


In [ ]:
# Interruptores
DESCARGAR_DATASET = False   # True solo para reentrenar o evaluar en test (requiere el dataset)
SUBIR_A_GIT = False         # True para hacer commit y push de modelos/resultados

SEMILLA = 42
TAM = 224
MEDIA, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
CLASE_MALA = "rottenapples"   # clase positiva para las métricas

REPO = "/content/proyectoAprendizaje"
RUTA_DATOS = "/content/apples_dataset"                  # el dataset va fuera del repo
MODELO_PATH = Path(f"{REPO}/modelos/baseline_v1.pt")    # modelo que se usa para predecir
MODELO_NUEVO = Path(f"{REPO}/modelos/baseline_v2.pt")   # donde se guarda un reentrenamiento

def fijar_semilla(s=SEMILLA):
    random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
fijar_semilla()
print("Dispositivo:", dev)

Dispositivo: cpu


In [ ]:
def tf_eval():
    return transforms.Compose([
        transforms.Resize((TAM, TAM)),
        transforms.ToTensor(),
        transforms.Normalize(MEDIA, STD),
    ])

def tf_train():
    return tf_eval()   # baseline SIN aumento de datos

def crear_modelo(n_clases=2, congelar=True):
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    if congelar:
        for p in m.parameters():
            p.requires_grad = False
    m.fc = nn.Linear(m.fc.in_features, n_clases)
    return m

In [ ]:
@torch.no_grad()
def evaluar_loader(modelo, loader, clases):
    modelo.eval()
    idx_mala = clases.index(CLASE_MALA)
    tp = fp = fn = tn = 0
    for x, y in loader:
        x, y = x.to(dev), y.to(dev)
        pred = modelo(x).argmax(1)
        for p, t in zip(pred.tolist(), y.tolist()):
            if p == idx_mala and t == idx_mala: tp += 1
            elif p == idx_mala and t != idx_mala: fp += 1
            elif p != idx_mala and t == idx_mala: fn += 1
            else: tn += 1
    total = tp + fp + fn + tn
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {"accuracy": (tp + tn) / total if total else 0.0,
            "precision_mala": prec, "recall_mala": rec, "f1_mala": f1,
            "matriz_confusion": {"TP": tp, "FP": fp, "FN": fn, "TN": tn}, "n": total}

In [ ]:
#carga modelo guardado

ck = torch.load(MODELO_PATH, map_location=dev)
modelo = crear_modelo(len(ck["clases"]), congelar=False)
modelo.load_state_dict(ck["state_dict"])
modelo = modelo.to(dev).eval()
clases = ck["clases"]
print("Modelo cargado:", MODELO_PATH.name, "| Clases:", clases)

Modelo cargado: baseline_v1.pt | Clases: ['freshapples', 'rottenapples']


In [ ]:
#evalua la imagen suministrada

@torch.no_grad()
def predecir_imagen(ruta, umbral_mala=0.5):
    img = Image.open(ruta).convert("RGB")
    x = tf_eval()(img).unsqueeze(0).to(dev)
    t0 = time.time()
    prob = torch.softmax(modelo(x), dim=1)[0].cpu()
    ms = (time.time() - t0) * 1000
    p = {c: float(prob[i]) for i, c in enumerate(clases)}
    etiqueta = CLASE_MALA if p[CLASE_MALA] >= umbral_mala else next(c for c in clases if c != CLASE_MALA)
    return etiqueta, p[etiqueta], p, ms

from google.colab import files
from IPython.display import Markdown, display

# sube la imagen
subida = files.upload()
if not subida:
    print("No se subió ninguna imagen. Vuelve a ejecutar esta celda y elige un archivo.")
else:
    ruta = list(subida.keys())[0]
    etiqueta, conf, p, ms = predecir_imagen(ruta, umbral_mala=0.5)
    display(Markdown(f"**Manzana:** {etiqueta.upper()} (confianza {conf:.1%})"))
    display(Markdown("**Probabilidades:** " + ", ".join(f"{k}: {v:.1%}" for k, v in p.items())))
    display(Markdown(f"**Inferencia:** {ms:.1f} ms"))
    if conf < 0.6:
        print("Aviso: confianza baja, resultado incierto.")
    display(Image.open(ruta).resize((300, 300)))

No se subió ninguna imagen. Vuelve a ejecutar esta celda y elige un archivo.


##reentrenar y evaluar (requiere el dataset)

Pon `DESCARGAR_DATASET = True` en la configuración y ejecuta esta sección en orden.


In [ ]:
if DESCARGAR_DATASET:
    if not os.path.exists(RUTA_DATOS):
        if not os.path.exists("/content/apples_dataset.zip"):
            !pip install -q gdown
            !gdown 1wNny4KKgdkzarUb7xgyJZE_UjIyNmtxY -O /content/apples_dataset.zip
        !unzip -tq /content/apples_dataset.zip
        !unzip -q /content/apples_dataset.zip -d {RUTA_DATOS}
    # Si el zip traía una carpeta raíz, ajusta la ruta
    if (Path(RUTA_DATOS) / "apples_dataset" / "train").exists():
        RUTA_DATOS = str(Path(RUTA_DATOS) / "apples_dataset")
    print("RUTA_DATOS =", RUTA_DATOS)
    for split in ["train", "test"]:
        for c in sorted(p.name for p in (Path(RUTA_DATOS) / split).iterdir() if p.is_dir()):
            print(f"{split}/{c}: {len(list((Path(RUTA_DATOS) / split / c).glob('*')))}")
else:
    print("Dataset omitido (DESCARGAR_DATASET = False).")

Downloading...
From (original): https://drive.google.com/uc?id=1wNny4KKgdkzarUb7xgyJZE_UjIyNmtxY
From (redirected): https://drive.google.com/uc?id=1wNny4KKgdkzarUb7xgyJZE_UjIyNmtxY&confirm=t&uuid=1d8336a3-2612-47c9-8f14-b98c50017b9b
To: /content/apples_dataset.zip
100% 669M/669M [00:02<00:00, 229MB/s]
No errors detected in compressed data of /content/apples_dataset.zip.
RUTA_DATOS = /content/apples_dataset
train/freshapples: 1693
train/rottenapples: 2342
test/freshapples: 395
test/rottenapples: 601


In [ ]:
assert DESCARGAR_DATASET, "Activa DESCARGAR_DATASET = True y ejecuta la celda del dataset."
import json, random, time, hashlib, shutil
from pathlib import Path
...
EXT = {".jpg", ".jpeg", ".png"}
def hashes(split):
    return {hashlib.md5(p.read_bytes()).hexdigest(): p
            for p in (Path(RUTA_DATOS) / split).rglob("*") if p.suffix.lower() in EXT}

h_tr, h_te = hashes("train"), hashes("test")
dup = set(h_tr) & set(h_te)
print(f"Duplicados exactos train <-> test: {len(dup)}")
if dup:
    print("Revisa y elimina estos duplicados antes de continuar:", [str(h_te[h]) for h in list(dup)[:5]])
# Ojo: esto solo detecta copias idénticas. Versiones rotadas/aumentadas de una misma foto
# no se detectan por hash; revisa los nombres de archivo y documenta lo que encuentres.

val_dir = Path(RUTA_DATOS) / "val"
if not val_dir.exists():
    random.seed(SEMILLA)
    for c in sorted(p.name for p in (Path(RUTA_DATOS) / "train").iterdir() if p.is_dir()):
        origen = sorted((Path(RUTA_DATOS) / "train" / c).glob("*"))
        random.shuffle(origen)
        n_val = int(0.15 * len(origen))
        (val_dir / c).mkdir(parents=True, exist_ok=True)
        for p in origen[:n_val]:
            shutil.move(str(p), val_dir / c / p.name)
        print(f"{c}: {n_val} imágenes movidas a val")
else:
    print("val ya existe, no se vuelve a crear.")

Duplicados exactos train <-> test: 0
freshapples: 253 imágenes movidas a val
rottenapples: 351 imágenes movidas a val


In [ ]:
assert DESCARGAR_DATASET, "Activa DESCARGAR_DATASET = True y ejecuta la celda del dataset."

EPOCAS, BATCH, LR = 5, 32, 1e-3

ds_tr = datasets.ImageFolder(f"{RUTA_DATOS}/train", tf_train())
ds_va = datasets.ImageFolder(f"{RUTA_DATOS}/val", tf_eval())      # validación, NO test
clases_tr = ds_tr.classes
assert CLASE_MALA in clases_tr, f"Clases encontradas: {clases_tr}"
print(f"Clases: {clases_tr} | train={len(ds_tr)} val={len(ds_va)}")

dl_tr = torch.utils.data.DataLoader(ds_tr, batch_size=BATCH, shuffle=True, num_workers=2)
dl_va = torch.utils.data.DataLoader(ds_va, batch_size=BATCH, num_workers=2)

fijar_semilla()
m_ent = crear_modelo(len(clases_tr)).to(dev)
opt = torch.optim.Adam(m_ent.fc.parameters(), lr=LR)
perdida = nn.CrossEntropyLoss()

MODELO_NUEVO.parent.mkdir(parents=True, exist_ok=True)
mejor_f1, t0 = -1.0, time.time()
for ep in range(1, EPOCAS + 1):
    m_ent.train()
    total_loss = 0.0
    for x, y in dl_tr:
        x, y = x.to(dev), y.to(dev)
        opt.zero_grad()
        loss = perdida(m_ent(x), y)
        loss.backward()
        opt.step()
        total_loss += loss.item() * x.size(0)
    m = evaluar_loader(m_ent, dl_va, clases_tr)
    print(f"Época {ep}/{EPOCAS} | loss={total_loss/len(ds_tr):.4f} | "
          f"val acc={m['accuracy']:.3f} F1(mala)={m['f1_mala']:.3f} recall(mala)={m['recall_mala']:.3f}")
    if m["f1_mala"] >= mejor_f1:   # se elige con validación, nunca con test
        mejor_f1 = m["f1_mala"]
        torch.save({"state_dict": m_ent.state_dict(), "clases": clases_tr,
                    "epoca": ep, "lr": LR, "batch": BATCH}, MODELO_NUEVO)

print(f"Listo en {time.time()-t0:.1f}s | mejor F1 val = {mejor_f1:.3f} | guardado en {MODELO_NUEVO.name}")

Clases: ['freshapples', 'rottenapples'] | train=3431 val=604


KeyboardInterrupt: 

In [ ]:
assert DESCARGAR_DATASET, "Activa DESCARGAR_DATASET = True y ejecuta la celda del dataset."

# Evalúa en test el modelo reentrenado si existe; si no, el baseline_v1

MODELO_EVALUAR = MODELO_NUEVO if MODELO_NUEVO.exists() else MODELO_PATH
ck = torch.load(MODELO_EVALUAR, map_location=dev)
modelo = crear_modelo(len(ck["clases"]), congelar=False)
modelo.load_state_dict(ck["state_dict"])
modelo = modelo.to(dev).eval()
clases = ck["clases"]

ds_te = datasets.ImageFolder(f"{RUTA_DATOS}/test", tf_eval())
dl_te = torch.utils.data.DataLoader(ds_te, batch_size=32, num_workers=2)
res = evaluar_loader(modelo, dl_te, clases)
print("Modelo evaluado:", MODELO_EVALUAR.name)
print(json.dumps(res, indent=2, ensure_ascii=False))

salida = Path(f"{REPO}/resultados/{MODELO_EVALUAR.stem}_test.json")
salida.parent.mkdir(parents=True, exist_ok=True)
salida.write_text(json.dumps(res, indent=2, ensure_ascii=False))
print("Resultados guardados en", salida)